In [1]:
import pandas as pd

import requests

In [3]:
url = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/2266/p/all/d/v2266%2013"

dados = requests.get(url).json()

ipca_indice = pd.DataFrame(dados, )

In [5]:
df_raw = pd.DataFrame(dados)

# Primeira linha explica o que cada código significa
header = df_raw.iloc[0]

# Dados verdadeiros começam da linha 1 em diante
df = df_raw.iloc[1:].copy()

# Descobre automaticamente as colunas corretas
col_data_codigo = header[header == "Mês (Código)"].index[0]
col_mes = header[header == "Mês"].index[0]
col_valor = header[header == "Valor"].index[0]

ipca_indice = pd.DataFrame({
    "data_codigo": df[col_data_codigo],
    "mes": df[col_mes],
    "numero_indice": df[col_valor]
})

ipca_indice["data"] = pd.to_datetime(ipca_indice["data_codigo"], format="%Y%m")

ipca_indice["numero_indice"] = (
    ipca_indice["numero_indice"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

ipca_indice = ipca_indice[["data", "mes", "numero_indice"]]

In [7]:
ipca_indice = ipca_indice[ipca_indice['data'] > '2024-02-01']

In [9]:
ipca_indice["razao_ni"] = (
    ipca_indice["numero_indice"] /
    ipca_indice["numero_indice"].shift(1)
)

In [11]:
ipca_indice

,data,mes,numero_indice,razao_ni
532,2024-03-01,março 2024,6869.14,NaN
533,2024-04-01,abril 2024,6895.24,1.003800
534,2024-05-01,maio 2024,6926.96,1.004600
535,2024-06-01,junho 2024,6941.51,1.002100
536,2024-07-01,julho 2024,6967.89,1.003800
537,2024-08-01,agosto 2024,6966.50,0.999801
538,2024-09-01,setembro 2024,6997.15,1.004400
539,2024-10-01,outubro 2024,7036.33,1.005599
540,2024-11-01,novembro 2024,7063.77,1.003900
541,2024-12-01,dezembro 2024,7100.50,1.005200


In [21]:
ipca_indice.to_excel('../data/raw/df_ipca.xlsx')